# Inference Demo
Load a trained checkpoint and generate text. Works on consumer GPUs (tested on RTX 5080 16GB).

**Usage:** Set `MODEL_DIR` and `CHECKPOINT` below, then Run All.

In [1]:
MODEL_DIR = "baseline"                              # directory containing model.py
CHECKPOINT = "baseline/runs/457654/baseline.pt"      # path to .pt file

In [2]:
import importlib.util, sys, os, math, torch
import torch.nn.functional as F
from transformers import AutoTokenizer

# dynamic import of model.py
root = os.path.dirname(os.path.abspath("."))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
spec = importlib.util.spec_from_file_location("model", os.path.join(MODEL_DIR, "model.py"))
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)

CONFIG = model_module.CONFIG
get_pos_encoding = model_module.get_pos_encoding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'})")

Device: cuda (NVIDIA GeForce RTX 5080)


In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

net = model_module.attention_net(
    tokenizer.vocab_size, CONFIG["hidden_size"], CONFIG["num_heads"],
    CONFIG["num_blocks"], CONFIG["seq_length"], CONFIG["dropout"]
)
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
net.load_state_dict(checkpoint["model_state_dict"])
net = net.to(device).eval()

print(f"Loaded {CONFIG['model']} from epoch {checkpoint.get('epoch', '?')}")
print(f"Eval perplexity at save: {math.exp(checkpoint['eval_loss']):.2f}")
print(f"Parameters: {sum(p.numel() for p in net.parameters()):,}")

Loaded baseline from epoch 83
Eval perplexity at save: 38.23
Parameters: 123,625,728


In [4]:
@torch.no_grad()
def generate(prompt, max_new_tokens=150, temperature=1.0, top_k=0, top_p=0.0):
    """
    Autoregressive generation.
      temperature=0 or (top_k=0 and top_p=0) -> greedy
      top_k > 0                               -> top-k sampling
      top_p > 0                                -> nucleus sampling
    """
    token_ids = tokenizer.encode(prompt)
    seq_length = CONFIG["seq_length"]
    hidden_size = CONFIG["hidden_size"]

    for _ in range(max_new_tokens):
        context = token_ids[-seq_length:]
        x = torch.LongTensor(context).unsqueeze(1).to(device)  # (ctx_len, 1)
        pos = get_pos_encoding(x.size(0), hidden_size, device)

        with torch.autocast(device.type, dtype=torch.bfloat16):
            scores = net(x, pos)
        logits = scores[-1, 0, :].float()

        if temperature != 1.0 and temperature > 0:
            logits = logits / temperature

        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[-1]] = float('-inf')

        if top_p > 0.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=0), dim=0)
            remove_mask = cumulative_probs - F.softmax(sorted_logits, dim=0) >= top_p
            sorted_logits[remove_mask] = float('-inf')
            logits = sorted_logits.scatter(0, sorted_idx, sorted_logits)

        probs = F.softmax(logits, dim=0)

        if temperature == 0 or (top_k == 0 and top_p == 0.0):
            next_token = torch.argmax(probs).item()
        else:
            next_token = torch.multinomial(probs, 1).item()

        if next_token == tokenizer.eos_token_id:
            break
        token_ids.append(next_token)

    return tokenizer.decode(token_ids)

In [5]:
prompts = [
    "The theory of relativity states that",
    "In a shocking turn of events, the",
    "def fibonacci(n):",
]

for p in prompts:
    print("=" * 70)
    print(f"PROMPT: {p}")
    print("-" * 70)

    print("\n[Greedy]")
    print(generate(p))

    print("\n[Top-k=50, temp=0.8]")
    print(generate(p, temperature=0.8, top_k=50))

    print("\n[Nucleus p=0.9, temp=0.9]")
    print(generate(p, temperature=0.9, top_p=0.9))
    print()

PROMPT: The theory of relativity states that
----------------------------------------------------------------------

[Greedy]
The theory of relativity states that the universe is a complex system of events that are not only observable but also observable.
The theory of relativity states that the universe is a complex system of events that are not observable but observable. The theory of relativity states that the universe is a complex system of events that are not observable but observable.
The theory of relativity states that the universe is a complex system of events that are not observable but observable. The theory of relativity states that the universe is a complex system of events that are not observable but observable.
The theory of relativity states that the universe is a complex system of events that are not observable but observable. The theory of relativity states that the universe is a complex system of events that are not observable but observable.
The theory of

[Top-k=50

In [7]:
# Try your own prompts here
print(generate("Nietzsche was a strong proponent of ", max_new_tokens=200, temperature=1, top_p=0.8))

Nietzsche was a strong proponent of vernacularism and a radical reformer. They formulated a form of vernacularism as a cause of vernacularism, and in some cases, vernacularism was an expression of a higher vernacularism. vernacularism was an expression of a lower vernacularism, and vernacularism was a form of vernacularism. vernacularism became an expression of a higher vernacularism, and vernacularism was a form of vernacularism. vernacularism was a form of vernacularism. vernacularism was the work of vernacularism in which an individual vernacularism was the work of vernacularism. vernacularism was the work of vernacularism in which a person vernacularism was the work of vernacularism. vernacularism was the work of vernacularism in which an individual vernacularism was the work of
